In [1]:
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt

from time import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from collections import OrderedDict

# Load custom modules
from common.functions import *
from common.gradient import *
from common.layers import *
from common.multi_layer_net_extend import MultiLayerNetExtend
from common.multi_layer_net import MultiLayerNet
from common.optimizer import *
from common.trainer import Trainer
from common.util import *

# 한글 폰트 및 마이너스 기호 표시 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [3]:
torch.cuda.init()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.reset_peak_memory_stats(device=None)
print("현재 디바이스:", device)

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

현재 디바이스: cuda


In [4]:
# CIFAR-100 데이터 로드 함수
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

data_path = './cifar-100-python'
# 데이터 경로 설정
train_path = os.path.join(data_path, 'train')
test_path = os.path.join(data_path, 'test')

# 학습 데이터 로드
train_data = unpickle(train_path)
test_data = unpickle(test_path)

# 메타데이터 로드 (클래스 이름 등)
meta_path = os.path.join(data_path, 'meta')
meta_data = unpickle(meta_path)

fine_label_names = [name.decode() for name in meta_data[b'fine_label_names']]
coarse_label_names = [name.decode() for name in meta_data[b'coarse_label_names']]

# 데이터 구조 확인
print("학습 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in train_data.keys()])
print("테스트 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in test_data.keys()])
print("메타 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in meta_data.keys()])

학습 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
테스트 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
메타 데이터 키: ['fine_label_names', 'coarse_label_names']


In [5]:
def one_hot_encode(y, num_classes):
    return np.eye(num_classes)[y.astype(int)]

# train set 으로 validation set 분할
x = train_data[b'data']
x = x.reshape(-1, 3, 32, 32)
t = np.array(train_data[b'coarse_labels'])
x_train, x_val, y_train, y_val = train_test_split(x, t, test_size=0.2, random_state=42, stratify=t)

# test set 정의
x_test = test_data[b'data']
x_test = x_test.reshape(-1, 3, 32, 32)
y_test = np.array(test_data[b'coarse_labels'])

x_test = x_test.astype(np.float32) / 255.0
x_train = x_train.astype(np.float32) / 255.0
x_val = x_val.astype(np.float32) / 255.0

y_train = one_hot_encode(y_train, 20)
y_val = one_hot_encode(y_val, 20)
y_test = one_hot_encode(y_test, 20)

x_train.shape, x_val.shape, x_test.shape, y_train.shape, y_val.shape, y_test.shape

((40000, 3, 32, 32),
 (10000, 3, 32, 32),
 (10000, 3, 32, 32),
 (40000, 20),
 (10000, 20),
 (10000, 20))

In [16]:
class CIFAR100Dataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        return torch.tensor(image), torch.tensor(label)

In [17]:
train_dataset = CIFAR100Dataset(x_train, y_train)
val_dataset = CIFAR100Dataset(x_val, y_val)
test_dataset = CIFAR100Dataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [46]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Convolutional Block 1 (Inspired by Keras example)
        self.conv1_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1) # padding='same'
        self.relu1_1 = nn.ReLU()
        self.conv1_2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) # padding='same'
        self.relu1_2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # (32, 16, 16)

        # Convolutional Block 2 (Inspired by Keras example)
        self.conv2_1 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) # padding='same'
        self.relu2_1 = nn.ReLU()
        self.conv2_2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1) # padding='same'
        self.relu2_2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # (128, 8, 8)
        
        # Note: Keras example has one more Conv2D(128, (3,3)) after pooling, 
        # but for 20 classes and to keep it slightly simpler, we'll go to FC layers.
        # If needed, that layer can be added:
        # self.conv3_1 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        # self.relu3_1 = nn.ReLU()
        # self.flattened_size = 128 * 8 * 8 (if conv3_1 is added and no pooling after)

        self.flattened_size = 128 * 8 * 8 # 8192
        
        # Dense Block 1
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.relu_fc1 = nn.ReLU()
        self.bn_fc1 = nn.BatchNorm1d(256)
        self.drop_fc1 = nn.Dropout(p=0.3)

        # Dense Block 2
        self.fc2 = nn.Linear(256, 256)
        self.relu_fc2 = nn.ReLU()
        self.bn_fc2 = nn.BatchNorm1d(256)
        self.drop_fc2 = nn.Dropout(p=0.3)

        # Output Layer (for 20 coarse labels)
        self.fc_out = nn.Linear(256, 20)

    def forward(self, x):
        # Conv Block 1
        x = self.relu1_1(self.conv1_1(x))
        x = self.relu1_2(self.conv1_2(x))
        x = self.pool1(x)
        
        # Conv Block 2
        x = self.relu2_1(self.conv2_1(x))
        x = self.relu2_2(self.conv2_2(x))
        x = self.pool2(x)
        
        # Flatten
        x = x.view(-1, self.flattened_size)
        
        # Dense Block 1
        x = self.fc1(x)
        x = self.relu_fc1(x)
        x = self.bn_fc1(x)
        x = self.drop_fc1(x)
        
        # Dense Block 2
        x = self.fc2(x)
        x = self.relu_fc2(x)
        x = self.bn_fc2(x)
        x = self.drop_fc2(x)
        
        # Output Layer
        x = self.fc_out(x)
        return x

In [47]:
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [48]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        target_labels = torch.max(labels, 1)[1] 
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, target_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted_train = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted_train == target_labels).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100 * correct_train / total_train
    
    # --- 검증 단계 ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            target_labels = torch.max(labels, 1)[1]
            
            outputs = model(images)
            loss = criterion(outputs, target_labels)
            val_loss += loss.item() * images.size(0)
            
            _, predicted_val = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted_val == target_labels).sum().item()
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = 100 * correct_val / total_val
    
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, '
          f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')

# --- 최종 테스트 단계 ---
model.eval()
correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in test_loader: # test_loader 사용
        images, labels = images.to(device), labels.to(device)
        target_labels = torch.max(labels, 1)[1]
        
        outputs = model(images)
        _, predicted_test = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted_test == target_labels).sum().item()

final_test_acc = 100 * correct_test / total_test
print(f'Test Accuracy on {total_test} test images: {final_test_acc:.2f}%')

Epoch [1/50], Train Loss: 2.3955, Train Acc: 26.45%, Val Loss: 2.1704, Val Acc: 32.73%
Epoch [2/50], Train Loss: 1.9677, Train Acc: 38.77%, Val Loss: 1.8898, Val Acc: 40.05%
Epoch [3/50], Train Loss: 1.7287, Train Acc: 45.74%, Val Loss: 1.7708, Val Acc: 44.33%
Epoch [4/50], Train Loss: 1.5925, Train Acc: 49.35%, Val Loss: 1.7026, Val Acc: 46.51%
Epoch [5/50], Train Loss: 1.4743, Train Acc: 53.26%, Val Loss: 1.7447, Val Acc: 46.11%
Epoch [6/50], Train Loss: 1.3718, Train Acc: 56.30%, Val Loss: 1.6207, Val Acc: 49.58%
Epoch [7/50], Train Loss: 1.2695, Train Acc: 59.26%, Val Loss: 1.5200, Val Acc: 52.83%
Epoch [8/50], Train Loss: 1.1812, Train Acc: 62.30%, Val Loss: 1.4497, Val Acc: 54.80%
Epoch [9/50], Train Loss: 1.0855, Train Acc: 65.16%, Val Loss: 1.5147, Val Acc: 54.37%
Epoch [10/50], Train Loss: 0.9934, Train Acc: 67.90%, Val Loss: 1.6004, Val Acc: 52.48%
Epoch [11/50], Train Loss: 0.9128, Train Acc: 70.36%, Val Loss: 1.6107, Val Acc: 53.40%
Epoch [12/50], Train Loss: 0.8373, Train 